# Publication Figures

Generate clean, publication-ready figures from all experimental results.
Assumes notebooks 01-05 have been run and results saved in `outputs/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

from persona_steering.config import VECTORS_DIR, FIGURES_DIR, Trait
from persona_steering.personas import load_all_personas, load_axis_personas
from persona_steering.analysis import (
    build_transfer_matrix,
    build_per_trait_transfer,
    decompose_shared_specific,
    compute_curvature,
)
from persona_steering.utils import load_pickle, ensure_output_dirs, cosine_similarity

ensure_output_dirs()

# Publication style
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

In [ ]:
# Load data
all_vectors = load_pickle(VECTORS_DIR / "all_vectors.pkl")
all_personas = load_all_personas()
axis_personas = load_axis_personas()
axis_slugs = [p.slug for p in axis_personas]
axis_positions = {p.slug: p.position for p in axis_personas}
all_slugs = [p.slug for p in all_personas]
traits = list(Trait)

sample_trait = traits[0]
sample_layers = sorted(all_vectors[axis_slugs[0]][sample_trait].keys())
mid_layer = sample_layers[len(sample_layers) // 2]

print(f"Personas: {[p.name for p in all_personas]}")
print(f"Axis: {[p.name for p in axis_personas]}")
print(f"Layer: {mid_layer}")

In [ ]:
# Figure 1: Transfer matrix heatmap (axis personas)
matrix = build_transfer_matrix(all_vectors, axis_slugs, traits, mid_layer)
labels = [f"{p.name}\n({p.position:+.1f})" for p in axis_personas]

fig, ax = plt.subplots(figsize=(5.5, 5))
sns.heatmap(matrix, annot=True, fmt=".2f", xticklabels=labels,
            yticklabels=labels, cmap="viridis", vmin=0, vmax=1, ax=ax,
            square=True, linewidths=0.5)
ax.set_title("Cross-Persona Steering Vector Similarity")
plt.savefig(FIGURES_DIR / "fig1_transfer_matrix.pdf")
plt.savefig(FIGURES_DIR / "fig1_transfer_matrix.png")
plt.show()

In [ ]:
# Figure 2: Per-trait similarity breakdown
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, trait in zip(axes.flat, traits):
    m = build_per_trait_transfer(all_vectors, axis_slugs, trait, mid_layer)
    sns.heatmap(m, annot=True, fmt=".2f",
                xticklabels=[p.name for p in axis_personas],
                yticklabels=[p.name for p in axis_personas],
                cmap="viridis", vmin=0, vmax=1, ax=ax, square=True, linewidths=0.5)
    ax.set_title(trait.value.title())

plt.suptitle("Per-Trait Cross-Persona Similarity", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig2_per_trait_transfer.pdf")
plt.savefig(FIGURES_DIR / "fig2_per_trait_transfer.png")
plt.show()

In [ ]:
# Figure 3: Similarity decay with axis distance
fig, ax = plt.subplots(figsize=(6, 4))
colors = plt.cm.Set2(np.linspace(0, 0.8, len(traits)))

for trait, color in zip(traits, colors):
    dists, sims = [], []
    for i, pa in enumerate(axis_slugs):
        for j, pb in enumerate(axis_slugs):
            if j <= i:
                continue
            va = all_vectors.get(pa, {}).get(trait, {}).get(mid_layer)
            vb = all_vectors.get(pb, {}).get(trait, {}).get(mid_layer)
            if va and vb:
                dists.append(abs(axis_positions[pa] - axis_positions[pb]))
                sims.append(cosine_similarity(va.vector, vb.vector))
    ax.scatter(dists, sims, color=color, label=trait.value, alpha=0.8, s=50, edgecolors='white', linewidth=0.5)

ax.set_xlabel("Axis Distance")
ax.set_ylabel("Cosine Similarity")
ax.set_title("Steering Similarity vs Persona Distance")
ax.legend(frameon=True, fancybox=False)
ax.grid(True, alpha=0.2)
plt.savefig(FIGURES_DIR / "fig3_decay_curve.pdf")
plt.savefig(FIGURES_DIR / "fig3_decay_curve.png")
plt.show()

In [ ]:
# Figure 4: Shared vs persona-specific components
shared_data = []
for trait in traits:
    vecs = {slug: all_vectors[slug][trait][mid_layer] for slug in axis_slugs
            if trait in all_vectors.get(slug, {}) and mid_layer in all_vectors[slug].get(trait, {})}
    if len(vecs) >= 2:
        decomp = decompose_shared_specific(vecs)
        shared_data.append({
            "trait": trait.value,
            "variance_explained": decomp.variance_explained,
        })

df_shared = pd.DataFrame(shared_data)

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(df_shared["trait"], df_shared["variance_explained"], color="steelblue", edgecolor='white')
ax.set_ylabel("Shared Variance Fraction")
ax.set_title("Shared vs Persona-Specific Steering")
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
plt.xticks(rotation=45, ha="right")
plt.savefig(FIGURES_DIR / "fig4_shared_specific.pdf")
plt.savefig(FIGURES_DIR / "fig4_shared_specific.png")
plt.show()

In [ ]:
# Figure 5: Layer-wise similarity curves
fig, ax = plt.subplots(figsize=(7, 4.5))
colors = plt.cm.Set2(np.linspace(0, 0.8, len(traits)))

for trait, color in zip(traits, colors):
    sims_by_layer = []
    for layer in sample_layers:
        layer_sims = []
        for i, pa in enumerate(axis_slugs):
            for j, pb in enumerate(axis_slugs):
                if j <= i:
                    continue
                va = all_vectors.get(pa, {}).get(trait, {}).get(layer)
                vb = all_vectors.get(pb, {}).get(trait, {}).get(layer)
                if va is not None and vb is not None:
                    layer_sims.append(cosine_similarity(va.vector, vb.vector))
        sims_by_layer.append(np.mean(layer_sims) if layer_sims else 0)

    ax.plot(sample_layers, sims_by_layer, marker="o", markersize=3,
            label=trait.value, color=color)

ax.set_xlabel("Layer")
ax.set_ylabel("Mean Cross-Persona Cosine Similarity")
ax.set_title("Layer-wise Cross-Persona Steering Similarity")
ax.legend(loc="lower right", frameon=True, fancybox=False)
ax.grid(True, alpha=0.2)
plt.savefig(FIGURES_DIR / "fig5_layerwise.pdf")
plt.savefig(FIGURES_DIR / "fig5_layerwise.png")
plt.show()

In [ ]:
# Figure 6: Our contrastive vectors vs reference trait vectors (Lu et al.)
from persona_steering.reference import load_trait_vector, load_assistant_axis, TRAIT_NAME_MAP

ref_data = []
for trait in traits:
    ref_vec = load_trait_vector(trait.value, device="cpu")
    ref_at_layer = ref_vec[mid_layer]

    for slug in all_slugs:
        our_vec = all_vectors.get(slug, {}).get(trait, {}).get(mid_layer)
        if our_vec is None:
            continue
        sim = cosine_similarity(our_vec.vector.cpu(), ref_at_layer)
        persona = next(p for p in all_personas if p.slug == slug)
        pos = persona.position if persona.position is not None else float("nan")
        ref_data.append({
            "trait": trait.value,
            "persona": persona.name,
            "position": pos,
            "cos_vs_ref": float(sim),
        })

df_ref = pd.DataFrame(ref_data)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [3, 2]})

# Panel A: heatmap
ax = axes[0]
pivot = df_ref.pivot_table(values="cos_vs_ref", index="trait", columns="persona")
col_order = [p.name for p in axis_personas if p.name in pivot.columns]
base_cols = [c for c in pivot.columns if c not in col_order]
pivot = pivot[base_cols + col_order]
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
            linewidths=0.5)
ax.set_title("A) Cosine Sim: Ours vs Reference")

# Panel B: axis-position trend for each trait (axis personas only)
ax = axes[1]
df_axis = df_ref[~df_ref["position"].isna()]
colors = plt.cm.Set2(np.linspace(0, 0.8, len(traits)))
for trait, color in zip(traits, colors):
    sub = df_axis[df_axis["trait"] == trait.value].sort_values("position")
    ax.plot(sub["position"], sub["cos_vs_ref"], marker="o", markersize=5,
            label=trait.value, color=color)

ax.set_xlabel("Axis Position")
ax.set_ylabel("Cosine Sim vs Reference")
ax.set_title("B) Reference Alignment by Position")
ax.legend(loc="best", frameon=True, fancybox=False, fontsize=8)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig6_reference_comparison.pdf")
plt.savefig(FIGURES_DIR / "fig6_reference_comparison.png")
plt.show()